In [1]:
import polars as pl
from datasets import load_dataset, load_from_disk
from replay.metrics import Recall, Precision, HitRate
import pandas as pd

In [2]:
item2item_recs = pl.read_parquet("../item2item/data/best_item2item_recs.parquet")
user2vec_recs = pl.read_parquet("../user2vec/data/user2vec_recs_v2.parquet")
ials_recs = pl.read_parquet("../ials/data/ials_recs_v2.parquet")
neural_item2item_recs = pl.read_parquet("../neural_item2item/data/item2item_recs.parquet")
co_liked_recs = pl.read_parquet("../item_co_occurance/co_liked_recs_v2.parquet")

In [31]:
all_recs = (
    item2item_recs
    .join(
        user2vec_recs,
        on="user_id",
        how="inner"
    )
    .join(
        ials_recs,
        on="user_id",
        how="inner"
    )
    .join(
        neural_item2item_recs,
        on="user_id",
        how="inner"
    )
    .join(
        co_liked_recs,
        on="user_id",
        how="inner"
    )
)

In [19]:
all_recs.head(5)

user_id,last_clicks,item2item_recs,user2vec_recs,ials_recs,neural_item2item_recs,co_liked_recs
i64,list[i64],list[i64],list[i64],list[i64],list[i64],list[i64]
14093892,"[234652449, 204938610, 157844389]","[234652449, 24725041, … 25890823]","[117277008, 171686736, … 84356]","[167392398, 3432615, … 53981912]","[52545540, 21126704, … 224856720]",[]
43635824,"[190925215, 62715930, … 186465714]","[190925215, 86058273, … 70295544]","[134898986, 143420921, … 166868709]","[34096337, 82817719, … 186724001]","[239138284, 186924312, … 223984876]","[167828822, 187498156, … 25652753]"
51312652,"[78187513, 154325377, … 224023613]","[219222831, 78187513, … 125002945]","[241574441, 1078426, … 174717899]","[41015722, 207254134, … 25607076]","[95956891, 12397437, … 201331724]","[134568799, 211966411, … 120280221]"
26806012,"[119852251, 10760442, 152335420]","[119852251, 58458760, … 182286971]","[152335420, 213949254, … 105830415]","[142049504, 164604147, … 189326168]","[51392190, 57042515, … 97127944]",[]
28921764,"[231708087, 172347039, … 11342126]","[231708087, 120656215, … 118163926]","[169859264, 195154229, … 76496176]","[84510695, 119653164, … 157248909]","[163333485, 72951261, … 196023342]",[]


In [32]:
def join_recs(row):
    joined_recs = set()
    for key in row:
        if "recs" in key:
            joined_recs = joined_recs | set(row[key])
    return list(joined_recs)

joined_recs = (
    all_recs
    .with_columns(
        pl.struct(["user2vec_recs", "item2item_recs", "ials_recs", "co_liked_recs", "neural_item2item_recs"]).apply(join_recs).alias("recs")
    )
    .select("user_id", "recs")
)

In [24]:
joined_recs.write_parquet("all_candidatates_recs.parquet")

In [42]:
DATA_PATH = "/home/jupyter/filestore/storage/datasets/user_clicks_20230501"

dataset = load_from_disk(DATA_PATH)

In [39]:
polars_ds = dataset.to_polars()

In [7]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")

train_items = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
    .select("item_id")
    .unique()["item_id"]
    .to_list()
)

In [7]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

test_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("future_clicks"))
    .join(
        joined_recs,
        on="user_id",
        how="inner"
    )
    .with_columns(pl.col("recs").apply(len).alias("recs_count"))
)

In [8]:
recs_stats = (
    test_interactions
    .with_columns(pl.col("recs").apply(len).alias("recs_count"))
    .select(
        pl.min("recs_count").alias("min_recs_count"),
        pl.mean("recs_count").alias("mean_recs_count"),
        pl.max("recs_count").alias("max_recs_count"),
    )
)

recs_stats

min_recs_count,mean_recs_count,max_recs_count
i64,f64,i64
756,1454.908481,3973


In [15]:
TOP_K_VALUES = [10, 100, 3973]

def calc_recall(row):
    return Recall._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_precision(row):
    return Precision._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_hitrate(row):
    return HitRate._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def intersection(row):
    return len(set(row["recs"]) & set(row["future_clicks"]))

metrics = (
    test_interactions
    .with_columns(
        pl.struct(["future_clicks", "recs"]).apply(calc_recall).alias("recall"),
        pl.struct(["future_clicks", "recs"]).apply(calc_precision).alias("precision"),
        pl.struct(["future_clicks", "recs"]).apply(calc_hitrate).alias("hitrate"),
    )
    #.filter(pl.col("hitrate").arr.get(2) == 1)
    .with_columns(pl.col("future_clicks").apply(len).alias("future_size"))
    #.filter(pl.col("future_size") > 10)
    .select(
        pl.col("recall").arr.get(0).mean().alias("recall@10"),
        pl.col("recall").arr.get(1).mean().alias("recall@100"),
        pl.col("recall").arr.get(2).mean().alias("recall@1000"),
        pl.col("precision").arr.get(0).mean().alias("precision@10"),
        pl.col("precision").arr.get(1).mean().alias("precision@100"),
        pl.col("precision").arr.get(2).mean().alias("precision@1000"),
        pl.col("hitrate").arr.get(0).mean().alias("hitrate@10"),
        pl.col("hitrate").arr.get(1).mean().alias("hitrate@100"),
        pl.col("hitrate").arr.get(2).mean().alias("hitrate@1000"),
        pl.col("hitrate").arr.get(0).sum().alias("hitrate_sum@10"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(1).sum().alias("hitrate_sum@100"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(2).sum().alias("hitrate_sum@1000") # количество рекомендаций, попавших в отложенную выборку
    )
    .head(5)
)

In [12]:
metrics # все пользователи ials v2

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.001482,0.014637,0.20322,0.002248,0.002184,0.000863,0.021563,0.15699,0.626554,2481.0,18063.0,72090.0


In [14]:
metrics # пользователи с более 5 кликами ials v2

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.001359,0.01256,0.186147,0.003403,0.003292,0.001317,0.032542,0.230015,0.789638,2300.0,16257.0,55810.0


In [17]:
metrics # пользователи с более 10 кликами ials v2

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.001201,0.011286,0.174333,0.004121,0.004005,0.001626,0.039257,0.271147,0.843748,2075.0,14332.0,44598.0


In [13]:
metrics # все пользователи

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.001771,0.017717,0.192791,0.002555,0.002509,0.000866,0.024405,0.174642,0.615298,2808.0,20094.0,70795.0


In [15]:
metrics # пользователи с более 5 кликами в будущем

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.00159,0.014702,0.175538,0.003843,0.003755,0.001321,0.036574,0.252879,0.779054,2585.0,17873.0,55062.0


In [17]:
metrics # пользователи с более 10 кликами в будущем

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.001377,0.013099,0.163925,0.004616,0.004549,0.001628,0.04376,0.295685,0.834951,2313.0,15629.0,44133.0


In [19]:
metrics # пользователи с более 15 кликами в будущем

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.001226,0.012155,0.155446,0.005207,0.005199,0.001882,0.049228,0.327784,0.86678,2076.0,13823.0,36553.0


In [21]:
metrics # пользователи у которых хотя бы одна релеватная рекомендация

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.002878,0.028794,0.313329,0.004153,0.004078,0.001407,0.039664,0.283834,1.0,2808.0,20094.0,70795.0


In [19]:
metrics # без neural item2item

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.001705,0.017079,0.12575,0.00249,0.002431,0.001929,0.023545,0.163066,0.50259,2709.0,18762.0,57827.0


In [14]:
metrics # добавили neural item2item

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.001532,0.015848,0.131056,0.002248,0.002223,0.002012,0.021433,0.15427,0.512316,2466.0,17750.0,58946.0
